## Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


## Dataset Extraction

In [ ]:
!pip install -q datasets

In [ ]:
import json
import random
import re
from pathlib import Path

import pandas as pd
import requests
from datasets import load_dataset, concatenate_datasets

SEED = 42
N_PER_DATASET = 250

MEDQA_PATH = "/content/drive/MyDrive/US_qbank.jsonl"
OUTPUT_PATH = "/content/drive/MyDrive/medical_qa_1000.jsonl"

random.seed(SEED)


def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, str):
        return x.strip()
    return str(x).strip()


def join_list_text(x):
    if x is None:
        return ""
    if isinstance(x, list):
        return " ".join([safe_str(i) for i in x if safe_str(i)])
    return safe_str(x)


def sample_list(data, n, seed=42):
    if len(data) < n:
        raise ValueError(f"Only {len(data)} rows available, cannot sample {n}.")
    rng = random.Random(seed)
    return rng.sample(data, n)


def dedup_rows(rows):
    dedup = []
    seen = set()
    for r in rows:
        key = (
            safe_str(r.get("dataset", "")),
            safe_str(r.get("question", "")),
            safe_str(r.get("answer", "")),
            safe_str(r.get("context", "")),
        )
        if key not in seen:
            seen.add(key)
            dedup.append(r)
    return dedup


# ====================================
# BioASQ parsing helpers
# ====================================
ANSWER_PATTERN = re.compile(
    r"<answer>\s*(.*?)\s*(?=<context>|$)",
    re.IGNORECASE | re.DOTALL
)

CONTEXT_PATTERN = re.compile(
    r"<context>\s*(.*)$",
    re.IGNORECASE | re.DOTALL
)


def parse_bioasq_text(text):
    text = safe_str(text)

    answer_match = ANSWER_PATTERN.search(text)
    context_match = CONTEXT_PATTERN.search(text)

    answer = answer_match.group(1).strip() if answer_match else ""
    context = context_match.group(1).strip() if context_match else ""

    return answer, context


# ====================================
# 1) MedQA-US from local jsonl
# question = <question> ... <options> ...
# answer   = answer
# context  = ""
# ====================================
def load_medqa_us_local(path, n=250, seed=42):
    rows = []
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            ex = json.loads(line)

            q = safe_str(ex.get("question", ""))
            answer_key = safe_str(ex.get("answer", ""))
            options = ex.get("options", {})

            if not q or not answer_key or not isinstance(options, dict) or len(options) == 0:
                continue

            option_lines = []
            for k, v in options.items():
                k = safe_str(k)
                v = safe_str(v)
                if k and v:
                    option_lines.append(f"{k}. {v}")

            answer_text = safe_str(options.get(answer_key, ""))
            final_answer = f"{answer_key}. {answer_text}" if answer_text else answer_key

            merged_question = f"<question> {q}\n<options>\n" + "\n".join(option_lines)

            rows.append({
                "dataset": "MedQA-US",
                "question": merged_question,
                "answer": answer_key,
                "context": ""
            })

    rows = dedup_rows(rows)
    return sample_list(rows, n, seed=seed)


# ====================================
# 2) PubMedQA from Hugging Face
# question = question only
# answer   = <long_answer> ... <final_decision> ...
# context  = context only
# ====================================
def load_pubmedqa(n=250, seed=42):
    ds = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")

    rows = []
    for ex in ds:
        q = safe_str(ex.get("question", ""))
        long_answer = safe_str(ex.get("long_answer", ""))
        final_decision = safe_str(ex.get("final_decision", ""))

        context_obj = ex.get("context", {})
        context_text = ""

        if isinstance(context_obj, dict):
            if "contexts" in context_obj:
                context_text = join_list_text(context_obj["contexts"])
            else:
                vals = []
                for v in context_obj.values():
                    if isinstance(v, list):
                        vals.extend(v)
                    else:
                        vals.append(v)
                context_text = join_list_text(vals)
        elif isinstance(context_obj, list):
            context_text = join_list_text(context_obj)
        else:
            context_text = safe_str(context_obj)

        merged_answer_parts = []
        if long_answer:
            merged_answer_parts.append(f"<long_answer> {long_answer}")
        if final_decision:
            merged_answer_parts.append(f"<final_decision> {final_decision}")
        merged_answer = "\n".join(merged_answer_parts)

        if q and (merged_answer or context_text):
            rows.append({
                "dataset": "PubMedQA",
                "question": q,
                "answer": merged_answer,
                "context": context_text
            })

    rows = dedup_rows(rows)
    return sample_list(rows, n, seed=seed)


# ====================================
# 3) BioASQ from Hugging Face
# question = question only
# answer   = content after <answer>
# context  = content after <context>
# ====================================
def load_bioasq(n=250, seed=42):
    train_ds = load_dataset("kroshan/BioASQ", split="train")
    valid_ds = load_dataset("kroshan/BioASQ", split="validation")
    ds = concatenate_datasets([train_ds, valid_ds])

    rows = []
    for ex in ds:
        q = safe_str(ex.get("question", ""))
        text = safe_str(ex.get("text", ""))

        answer, context = parse_bioasq_text(text)

        if q and (answer or context):
            rows.append({
                "dataset": "BioASQ",
                "question": q,
                "answer": answer,
                "context": context
            })

    rows = dedup_rows(rows)
    return sample_list(rows, n, seed=seed)


# ====================================
# 4) MediQA via datasets-server parquet API
# question = question only
# answer   = answer
# context  = ""
# ====================================
def get_mediqa_parquet_urls():
    api_url = "https://datasets-server.huggingface.co/parquet?dataset=bigbio/mediqa_qa"
    resp = requests.get(api_url, timeout=60)
    resp.raise_for_status()
    data = resp.json()

    parquet_files = data.get("parquet_files", [])
    if not parquet_files:
        raise RuntimeError("No parquet files returned for bigbio/mediqa_qa.")

    target = []
    for item in parquet_files:
        if item.get("config") == "mediqa_qa_bigbio_qa":
            target.append(item)

    if not target:
        raise RuntimeError("No parquet files found for config=mediqa_qa_bigbio_qa.")

    return target


def load_mediqa(n=250, seed=42):
    parquet_files = get_mediqa_parquet_urls()

    dfs = []
    for item in parquet_files:
        url = item["url"]
        split_name = item.get("split", "unknown")
        df = pd.read_parquet(url)
        df["__split__"] = split_name
        dfs.append(df)

    if not dfs:
        raise RuntimeError("Failed to read any MediQA parquet files.")

    df = pd.concat(dfs, ignore_index=True)

    rows = []
    for _, ex in df.iterrows():
        q = safe_str(ex.get("question", ""))
        ans_raw = ex.get("answer", "")

        if isinstance(ans_raw, list):
            ans = "\n".join([safe_str(a) for a in ans_raw if safe_str(a)])
        else:
            ans = safe_str(ans_raw)

        if q and ans:
            rows.append({
                "dataset": "MediQA",
                "question": q,
                "answer": ans,
                "context": ""
            })

    rows = dedup_rows(rows)
    return sample_list(rows, n, seed=seed)


# ====================================
# Main
# ====================================
def main():
    medqa_rows = load_medqa_us_local(MEDQA_PATH, n=N_PER_DATASET, seed=SEED)
    pubmedqa_rows = load_pubmedqa(n=N_PER_DATASET, seed=SEED)
    bioasq_rows = load_bioasq(n=N_PER_DATASET, seed=SEED)
    mediqa_rows = load_mediqa(n=N_PER_DATASET, seed=SEED)

    all_rows = medqa_rows + pubmedqa_rows + bioasq_rows + mediqa_rows

    rng = random.Random(SEED)
    rng.shuffle(all_rows)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        for row in all_rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"Saved {len(all_rows)} rows to: {OUTPUT_PATH}")

    counts = {}
    for r in all_rows:
        counts[r["dataset"]] = counts.get(r["dataset"], 0) + 1

    print("\nCounts by dataset:")
    for k, v in counts.items():
        print(f"{k}: {v}")

    print("\nFirst 5 examples:")
    for ex in all_rows[:5]:
        print(json.dumps(ex, ensure_ascii=False, indent=2))
        print("-" * 80)


main()

Saved 1000 rows to: /content/drive/MyDrive/medical_qa_1000.jsonl

Counts by dataset:
MediQA: 250
BioASQ: 250
MedQA-US: 250
PubMedQA: 250

First 5 examples:
{
  "dataset": "MediQA",
  "question": "what are autoimmune blood disorders",
  "answer": "[\"Autoimmune disorders: An autoimmune disorder occurs when the body's immune system attacks and destroys healthy body tissue by mistake. There are more than 80 types of autoimmune disorders. The blood cells in the body's immune system help protect against harmful substances. Examples include bacteria, viruses, toxins, cancer cells, and blood and tissue from outside the body. These substances contain antigens. The immune system produces antibodies against these antigens that enable it to destroy these harmful substances. When you have an autoimmune disorder, your immune system does not distinguish between healthy tissue and antigens. As a result, the body sets off a reaction that destroys normal tissues. The exact cause of autoimmune disorders